In [1]:
!pip install transformers datasets torch accelerate


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset

dataset = load_dataset("text", data_files="custom_data.txt")
dataset = dataset["train"].train_test_split(test_size=0.1)  # 90/10 train/validation split

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset

dataset = load_dataset("text", data_files="custom_data.txt")
dataset = dataset["train"].train_test_split(test_size=0.1)  # 90/10 train/validation split

In [4]:
from transformers import GPT2Tokenizer, AutoTokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Fix padding issue

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_tensors="pt")

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

Map: 100%|██████████| 1/1 [00:00<00:00, 225.95 examples/s]


In [5]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch

model = GPT2LMHeadModel.from_pretrained("gpt2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    # overwrite_output_dir=True, # Removed to fix TypeError, consider restarting runtime if issues persist
    num_train_epochs=3,  # 3 epochs for small data; monitor loss
    per_device_train_batch_size=4,  # Adjust based on GPU (Colab T4: 4-8)
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    # evaluation_strategy="steps", # Removed to fix TypeError, consider restarting runtime if issues persist
    # eval_steps=50, # Removed as it depends on evaluation_strategy
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)  # Causal LM, not masked

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

Loading weights: 100%|██████████| 148/148 [00:04<00:00, 35.69it/s, Materializing param=transformer.wte.weight]              


In [6]:
trainer.train()
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.94s/it]


('./gpt2-finetuned/tokenizer_config.json', './gpt2-finetuned/tokenizer.json')

In [7]:
from transformers import pipeline

generator = pipeline("text-generation", model="./gpt2-finetuned", tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

prompt = "Once upon a time"
result = generator(prompt, max_length=100, num_return_sequences=1, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(result[0]["generated_text"])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 168.99it/s, Materializing param=transformer.wte.weight]             
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length', 'num_return_sequences', 'pad_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time, I was in a position where I had to find a team to play with, and I was thinking that I could come up with something that I could play with and in one game, to show myself.

It was just a matter of doing it. But then I got a call that I have to sit down and get prepared for the season. I had a good game against Columbus and had some good minutes against them in my first game against Columbus. I thought I was going to play a good game against them. But then I went on a little bit off the charts and it took a little bit longer to get to the playoffs. That's when I was really looking for something to do.

So it's been a long time since I have been able to play three or four games in a row. But, I've put myself in the position where I can play for the team that has me and I can get a chance to prove myself. I've been in the top six of the American League pennant and I've been in the top six of the American League championship. And I've been in the top six of the American L